In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import shutil
import zipfile
from tqdm import tqdm
from pathlib import Path
from yolo_tools import get_yolo_label_df
from data_vis.yolo_vis import yolo_mdet_vis
from yolo2xanylabeling import yolo_to_xanylabeling_dir
from tqdm.notebook import tqdm
from yolo_tools import get_stem2img

In [ ]:
root_path = r'E:\data\202502_signboard\data_annotation\ps_data\task'
# class_file = r'E:\data\202502_signboard\data_annotation\docs\class.txt'
class_file = r'E:\data\202502_signboard\data_annotation\docs\class_c6.txt'
att_file = r'E:\data\202502_signboard\data_annotation\docs\attribute.yaml'
yolo_merge_dir = os.path.join(root_path, 'merge_dir_0822')
yolo_merge_image_dir = os.path.join(yolo_merge_dir, 'images')
yolo_merge_label_dir = os.path.join(yolo_merge_dir, 'labels')
defect_list = ['deformation', 'broken', 'abandonment', 'corrosion']
task_list = [
        "task_0806_0821",
        "task_0808_021",
        "task_0812_0821",
        # "task_0819_0821",
    ]
task_merge_list = [
        # 'task_0613_0714',
        # 'task_0620_0717',
        # 'task_0618_0717',
        # 'task_0625_0714',
        # 'task_0627_0714',
        # 'task_0702_0710',
        # 'task_0704_0710',
        # 'task_0709_0714',
        # 'task_0711_0714',
        # 'task_0709_0724',
        # 'task_0711_0724',
        # 'task_0716_0728',
        # 'task_0718_0730',
        # 'task_0722_0730',
        # 'task_0725_0805',
        # 'task_0610_0816',
        # 'task_0811_0816',
        "task_0806_0821",
        "task_0808_021",
        "task_0812_0821",
        # "task_0819_0821",
    ]

In [ ]:
# def unzip_file(zip_path):
#     print(f'{zip_path} unzip...')
#     with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#         zip_ref.extractall(zip_path.replace('.zip', ''))
#     print(f'{zip_path} done\n')

In [5]:
# for task_name in task_list:
#     input_label_zip = os.path.join(root_path, task_name+'.zip')
#     unzip_file(input_label_zip)

In [6]:
def att_check(input_dir):
    count = 0
    label_list = os.listdir(input_dir)
    for label_name in tqdm(label_list):
        label_path = os.path.join(input_dir, label_name)
        with open(label_path, 'r') as f:
            lines = f.readlines()
            for idx, line in enumerate(lines):
                if line[2] == '0':
                    lines[idx] = lines[idx][0:2] + '4' + lines[idx][3:]
                    count += 1
                else:
                    continue
        with open(label_path, 'w') as f:
            f.writelines(lines)
    print(f'change {count} lines')


  0%|          | 0/5486 [00:00<?, ?it/s]

change 139 lines


  0%|          | 0/717 [00:00<?, ?it/s]

change 759 lines


In [7]:
for task_name in task_list:
    # src_labels_dir = os.path.join(root_path, task_name, task_name, 'labels')
    src_labels_dir = os.path.join(root_path, task_name, task_name)
    att_check(src_labels_dir)

In [8]:
att_check(r'E:\data\202502_signboard\data_annotation\ps_data\task\task_0811_0816\task_0811_0816\labels')

已移动: cam_DA4930148_cam_DA4930148_20250610130200683.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130201685.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130202690.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130203695.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130204687.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130205682.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130206685.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130207682.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130208684.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130209683.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130210684.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130211684.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130212682.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130213682.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130214685.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130215699.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130216683.txt
已移动: cam_DA4930148_cam_DA4930148_20250610130217683.txt
已移动: cam_D

In [9]:
def move_labels(source_dir):
    # 定义目标文件夹路径
    labels_dir = os.path.join(source_dir, 'labels')

    # 创建labels文件夹（如果不存在）
    os.makedirs(labels_dir, exist_ok=True)

    # 定义图像文件扩展名
    image_extensions = ['.txt']

    # 统计移动的文件数量
    moved_count = 0

    # 遍历源文件夹中的所有文件
    for file_name in os.listdir(source_dir):
        # 获取文件扩展名
        _, ext = os.path.splitext(file_name)

        # 检查是否为图像文件
        if ext.lower() in image_extensions:
            # 构建完整的文件路径
            source_path = os.path.join(source_dir, file_name)
            target_path = os.path.join(labels_dir, file_name)

            # 移动文件
            shutil.move(source_path, target_path)
            print(f"已移动: {file_name}")
            moved_count += 1

    print(f"移动完成，共移动了 {moved_count} 个图像文件到 {labels_dir}")

In [10]:
for task_name in task_list:
    src_labels_dir = os.path.join(root_path, task_name, task_name)
    move_labels(src_labels_dir)

In [ ]:
def get_stem2img_dict(img_dir):
    img_list = [img_name for img_name in os.listdir(img_dir) if img_name.endswith('.jpg')]
    stem_list = [Path(img).stem for img in img_list]
    stem2img_dict = dict(zip(stem_list, img_list))
    return stem2img_dict

In [ ]:
def copy_all(input_label_dir, output_label_dir, input_image_dir, output_image_dir):
    os.makedirs(output_label_dir, exist_ok=True)
    os.makedirs(output_image_dir, exist_ok=True)
    label_file_list = os.listdir(input_label_dir)
    stem2img_dict = get_stem2img_dict(input_image_dir)
    for label_name in tqdm(label_file_list):
        input_label_path = os.path.join(input_label_dir, label_name)
        label_name_stem = Path(label_name).stem
        image_name = stem2img_dict[label_name_stem]
        input_image_path = os.path.join(input_image_dir, image_name)
        output_label_path = os.path.join(output_label_dir, label_name)
        output_image_path = os.path.join(output_image_dir, image_name)
        shutil.copy(input_label_path, output_label_path)
        shutil.copy(input_image_path, output_image_path)

In [16]:
def find_defect(input_label_dir, output_label_dir, input_image_dir, output_image_dir, defect_list=defect_list):
    os.makedirs(output_label_dir, exist_ok=True)
    os.makedirs(output_image_dir, exist_ok=True)
    label_file_list = os.listdir(input_label_dir)
    stem2img_dict = get_stem2img_dict(input_image_dir)
    for label_name in tqdm(label_file_list):
        input_label_path = os.path.join(input_label_dir, label_name)
        df = get_yolo_label_df(input_label_path, mdet=True, attributes=defect_list)
        with_defect = (df[defect_list] > 0).any().any()
        if with_defect:
            label_name_stem = Path(label_name).stem
            image_name = stem2img_dict[label_name_stem]
            input_image_path = os.path.join(input_image_dir, image_name)
            output_label_path = os.path.join(output_label_dir, label_name)
            output_image_path = os.path.join(output_image_dir, image_name)
            shutil.copy(input_label_path, output_label_path)
            shutil.copy(input_image_path, output_image_path)

  0%|          | 0/5486 [00:00<?, ?it/s]

110


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'Y:\\ZHL\\isds\\PS\\task0811\\merge_dir'

In [12]:
# def find_defect(input_label_dir, input_json_dir, output_label_dir, input_image_dir, output_image_dir, defect_list=defect_list):
#     os.makedirs(output_label_dir, exist_ok=True)
#     os.makedirs(output_image_dir, exist_ok=True)
#     label_file_list = os.listdir(input_label_dir)
#     stem2img_dict = get_stem2img_dict(input_image_dir)
#     for label_name in tqdm(label_file_list):
#         input_label_path = os.path.join(input_label_dir, label_name)
#         input_json_path = os.path.join(input_json_dir, label_name.replace('.txt', '.json'))
#         if not os.path.exists(input_json_path):
#             continue
#         df = get_yolo_label_df(input_label_path, mdet=True, attributes=defect_list)
#         with_defect = (df[defect_list] > 0).any().any()
#         if with_defect:
#             label_name_stem = Path(label_name).stem
#             image_name = stem2img_dict[label_name_stem]
#             input_image_path = os.path.join(input_image_dir, image_name)
#             output_label_path = os.path.join(output_label_dir, label_name)
#             output_image_path = os.path.join(output_image_dir, image_name)
#             shutil.copy(input_label_path, output_label_path)
#             shutil.copy(input_image_path, output_image_path)


  0%|          | 0/110 [00:00<?, ?it/s]

0610 data (110/110)


0it [00:00, ?it/s]

0811 data (0/110)


In [ ]:
# for task_name in task_list:
#     input_json_dir = os.path.join(root_path, task_name, task_name, 'json')
#     input_label_dir = os.path.join(root_path, task_name, task_name, 'labels')
#     input_image_dir = rf'Y:\ZHL\isds\PS\task{task_name.split("_")[1]}\merge_dir'
#     output_dir = os.path.join(root_path, task_name, 'yolo_data')
#     output_image_dir = os.path.join(output_dir, 'images')
#     output_label_dir = os.path.join(output_dir, 'labels')
#     find_defect(
#         input_label_dir,
#         input_json_dir,
#         output_label_dir,
#         input_image_dir,
#         output_image_dir,
#         defect_list,
#     )

In [ ]:
for task_name in task_list:
    input_label_dir = os.path.join(root_path, task_name, task_name, 'labels')
    input_image_dir = rf'Y:\ZHL\isds\PS\task{task_name.split("_")[1]}\merge_dir'
